## 1.Loading the Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from warnings import filterwarnings
filterwarnings('ignore')

In [ ]:
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,AdaBoostClassifier,GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from lightgbm import LGBMClassifier
from sklearn.svm import SVC

In [ ]:
from sklearn.metrics import accuracy_score

## 2.Loading the Dataset

In [ ]:
df_train=pd.read_csv('C:/Users/Sushree Jena/Desktop/imarticus/machine learning/vindhya/janatahack/Train.csv')

In [ ]:
df_train.head()

In [ ]:
df_test=pd.read_csv('C:/Users/Sushree Jena/Desktop/imarticus/machine learning/vindhya/janatahack/Test.csv')

In [ ]:
df_test.head()

## 3.Exploratory Data Analysis

We will combine both train and test data for the ease of performing EDA. At the time of model building, we will separate it.

In [ ]:
df_train['source']='train'
df_test['source']='test'

In [ ]:
df_cust=pd.concat([df_train,df_test],ignore_index=True)

In [ ]:
df_cust.head()

In [ ]:
df_cust.shape

In [ ]:
df_cust.info()

In [ ]:
df_cust.Graduated.value_counts()

In [ ]:
df_cust.Profession.value_counts()

In [ ]:
df_cust.Segmentation.value_counts()

In [ ]:
df_cust.describe()

In [ ]:
#Checking for outliers

In [ ]:
sns.boxplot(data=df_cust)
plt.show()

In [ ]:
#Converting the Age column into a categorical column

In [ ]:
df_cust['age_cat']=pd.cut(df_cust.Age,bins=[15,30,60,90],labels=['young','middle','old'])

In [ ]:
df_cust.age_cat.value_counts()

In [ ]:
#Dropping the Age column since we donot require it now.

df_cust=df_cust.drop('Age',axis=1)

In [ ]:
#Treating the outliers

In [ ]:
cols=['Work_Experience','Family_Size']

In [ ]:
def boundaries(data,variable):
    LL=data[variable].quantile(.25)-(((data[variable].quantile(.75))-(data[variable].quantile(.25)))*1.5)
    UL=data[variable].quantile(.75)+(((data[variable].quantile(.75))-(data[variable].quantile(.25)))*1.5)
    return LL,UL

In [ ]:
for col in cols:
    LL,UL=boundaries(df_cust,col)
    
    df_cust[col]=np.where(df_cust[col]<LL,LL,
                         np.where(df_cust[col]>UL,UL,df_cust[col]))

In [ ]:
sns.boxplot(data=df_cust)
plt.show()

We can see that the outliers have been treated now.

In [ ]:
#Checking for missing values

In [ ]:
df_cust.isnull().sum()

In [ ]:
df_cust.Ever_Married.value_counts()

We can find that we have missing values in Ever_married,Graduated,Profession,Work_Experience,Family_Size,Var_1 columns. We need to treat them before proceeding to model building.

In [ ]:
cols=['Ever_Married','Graduated','Profession','Var_1']

In [ ]:
for col in cols:
    df_cust[col]=df_cust[col].fillna(df_cust[col].mode()[0])

In [ ]:
df_cust.isnull().sum()

In [ ]:
#Filling the null values in Work_Experience based on their Profession

In [ ]:
missing_index=df_cust.loc[df_cust.Work_Experience.isnull(),['Profession','Work_Experience']]

In [ ]:
df1=df_cust.groupby('Profession')['Work_Experience'].mean().round(0)

In [ ]:
df_cust.loc[missing_index.index,'Work_Experience']=df_cust.loc[missing_index.index,'Profession'].apply(lambda x:df1.loc[x])

In [ ]:
#Filling the null values in Family_Size based on their marriage

In [ ]:
missing_index1=df_cust.loc[df_cust.Family_Size.isnull(),['Ever_Married','Family_Size']]

In [ ]:
df2=df_cust.groupby('Ever_Married')['Family_Size'].median()

In [ ]:
df_cust.loc[missing_index1.index,'Family_Size']=df_cust.loc[missing_index1.index,'Ever_Married'].apply(lambda x:df2.loc[x])

In [ ]:
df_cust.isnull().sum()

The missing value have been treated now.

In [ ]:
#Correlation

In [ ]:
vcorr=df_cust.corr()

In [ ]:
sns.heatmap(vcorr[(vcorr>0.85) | (vcorr<-0.85)],annot=True)
plt.show()

There is no strong correlation between the features.

In [ ]:
#Creating a backup file

In [ ]:
df_cust1=df_cust.copy(deep=True)

## 4.Separating the train and test set

In [ ]:
df_train=df_cust1.loc[df_cust1['source']=='train']
df_test=df_cust1.loc[df_cust1['source']=='test']

In [ ]:
df_train.shape,df_test.shape

In [ ]:
df_train1=df_train.drop(['ID','source'],axis=1)

In [ ]:
df_test1=df_test.drop(['source','ID','Segmentation'],axis=1)

In [ ]:
df_train1.shape,df_test1.shape

## 5.Splitting into X_train and y_train

In [ ]:
X_train=df_train1.drop('Segmentation',axis=1)

In [ ]:
y_train=df_train1['Segmentation']

In [ ]:
X_test=df_test1

## 6.Dummy Encoding

In [ ]:
X_train=pd.get_dummies(X_train,drop_first=True)

In [ ]:
X_train.shape

In [ ]:
X_test=pd.get_dummies(X_test,drop_first=True)

In [ ]:
X_test.shape

In [ ]:
le=LabelEncoder()

In [ ]:
y_train=le.fit_transform(y_train)

In [ ]:
y_train.shape

## 7.Scaling the data 

In [ ]:
sc=StandardScaler()

In [ ]:
X_train_scaled=sc.fit_transform(X_train)
X_test_scaled=sc.transform(X_test)

## 8.Model Building

1.Logistic Regression

In [ ]:
lr=LogisticRegression()

In [ ]:
lr.fit(X_train_scaled,y_train)

In [ ]:
y_pred_train=lr.predict(X_train_scaled)

In [ ]:
y_pred_test=lr.predict(X_test_scaled)

In [ ]:
print('Training Accuracy is: ',accuracy_score(y_train,y_pred_train))

In [ ]:
y_pred_test

In [ ]:
submit_lr=pd.DataFrame({'ID':df_test['ID'],'Segmentation':y_pred_test})

In [ ]:
def segment(i):
    if i==0:
        i='A'
    elif i==1:
        i='B'
    elif i==2:
        i='C'
    else:
        i='D'
    return i

In [ ]:
submit_lr['Segmentation']=submit_lr['Segmentation'].apply(segment)

In [ ]:
submit_lr.to_csv('janatahack.submit_lr.csv',index=False)

2.Decision Tree

In [ ]:
dt=DecisionTreeClassifier()

In [ ]:
dt.fit(X_train_scaled,y_train)

In [ ]:
y_pred_train=dt.predict(X_train_scaled)

In [ ]:
y_pred_test=dt.predict(X_test_scaled)

In [ ]:
print('Training Accuracy is: ',accuracy_score(y_train,y_pred_train))

In [ ]:
submit_dt=pd.DataFrame({'ID':df_test['ID'],'Segmentation':y_pred_test})

In [ ]:
submit_dt['Segmentation']=submit_dt['Segmentation'].apply(segment)

In [ ]:
submit_dt.to_csv('janatahack.submit_dt.csv',index=False)

3.Random Forest Classifier

In [ ]:
rf=RandomForestClassifier()

In [ ]:
rf.fit(X_train_scaled,y_train)

In [ ]:
y_pred_train=rf.predict(X_train_scaled)

In [ ]:
y_pred_test=rf.predict(X_test_scaled)

In [ ]:
print('Training Accuracy is: ',accuracy_score(y_train,y_pred_train))

In [ ]:
submit_rf=pd.DataFrame({'ID':df_test['ID'],'Segmentation':y_pred_test})

In [ ]:
submit_rf['Segmentation']=submit_rf['Segmentation'].apply(segment)

In [ ]:
submit_rf.to_csv('janatahack.submit_rf.csv',index=False)

4.GridSearchCV

In [ ]:
tuned_parameters={'n_estimators':[85,100],'min_samples_split':[15,20],'max_depth':[8,10]}

In [ ]:
gs_model=GridSearchCV(estimator=rf,param_grid=tuned_parameters,cv=5)
gs_model.fit(X_train_scaled,y_train)

In [ ]:
print('The best parameters are: ',gs_model.best_params_)

In [ ]:
#Rebuilding the model

In [ ]:
rf=RandomForestClassifier(n_estimators=100,min_samples_split=15,max_depth=10)

In [ ]:
rf.fit(X_train_scaled,y_train)

In [ ]:
y_pred_train=rf.predict(X_train_scaled)

In [ ]:
y_pred_test=rf.predict(X_test_scaled)

In [ ]:
print('Training Accuracy is: ',accuracy_score(y_train,y_pred_train))

In [ ]:
submit_GridSearchCV_rf=pd.DataFrame({'ID':df_test['ID'],'Segmentation':y_pred_test})

In [ ]:
submit_GridSearchCV_rf['Segmentation']=submit_GridSearchCV_rf['Segmentation'].apply(segment)

In [ ]:
submit_GridSearchCV_rf.to_csv('janatahack.submit_GridSearchCV_rf.csv',index=False)

5.XGBClassifier

In [ ]:
xgb=XGBClassifier()

In [ ]:
xgb.fit(X_train_scaled,y_train)

In [ ]:
y_pred_train=xgb.predict(X_train_scaled)

In [ ]:
y_pred_test=xgb.predict(X_test_scaled)

In [ ]:
print('Training Accuracy is: ',accuracy_score(y_train,y_pred_train))

In [ ]:
submit_xgb=pd.DataFrame({'ID':df_test['ID'],'Segmentation':y_pred_test})

In [ ]:
submit_xgb['Segmentation']=submit_xgb['Segmentation'].apply(segment)

In [ ]:
submit_xgb.to_csv('janatahack.submit_xgb.csv',index=False)

6.Gradient Boosting Classifier

In [ ]:
gb=GradientBoostingClassifier()

In [ ]:
gb.fit(X_train_scaled,y_train)

In [ ]:
y_pred_train=gb.predict(X_train_scaled)

In [ ]:
y_pred_test=gb.predict(X_test_scaled)

In [ ]:
print('Training Accuracy is: ',accuracy_score(y_train,y_pred_train))

In [ ]:
submit_gb=pd.DataFrame({'ID':df_test['ID'],'Segmentation':y_pred_test})

In [ ]:
submit_gb['Segmentation']=submit_gb['Segmentation'].apply(segment)

In [ ]:
submit_gb.to_csv('janatahack.submit_gb.csv',index=False)

7.AdaBoostClassifier

In [ ]:
ab=AdaBoostClassifier()

In [ ]:
ab.fit(X_train_scaled,y_train)

In [ ]:
y_pred_train=ab.predict(X_train_scaled)

In [ ]:
y_pred_test=ab.predict(X_test_scaled)

In [ ]:
print('Training Accuracy is: ',accuracy_score(y_train,y_pred_train))

In [ ]:
submit_ab=pd.DataFrame({'ID':df_test['ID'],'Segmentation':y_pred_test})

In [ ]:
submit_ab['Segmentation']=submit_ab['Segmentation'].apply(segment)

In [ ]:
submit_ab.to_csv('janatahack.submit_ab.csv',index=False)

8.LGBMClassifier

In [ ]:
lgbm=LGBMClassifier()

In [ ]:
lgbm.fit(X_train_scaled,y_train)

In [ ]:
y_pred_train=lgbm.predict(X_train_scaled)

In [ ]:
y_pred_test=lgbm.predict(X_test_scaled)

In [ ]:
print('Training Accuracy is: ',accuracy_score(y_train,y_pred_train))

In [ ]:
submit_lgbm=pd.DataFrame({'ID':df_test['ID'],'Segmentation':y_pred_test})

In [ ]:
submit_lgbm['Segmentation']=submit_lgbm['Segmentation'].apply(segment)

In [ ]:
submit_lgbm.to_csv('janatahack.submit_lgbm.csv',index=False)

9.Naive Bayes

In [ ]:
gnb=GaussianNB()

In [ ]:
gnb.fit(X_train_scaled,y_train)

In [ ]:
y_pred_train=gnb.predict(X_train_scaled)

In [ ]:
y_pred_test=gnb.predict(X_test_scaled)

In [ ]:
print('Training Accuracy is: ',accuracy_score(y_train,y_pred_train))

In [ ]:
submit_gnb=pd.DataFrame({'ID':df_test['ID'],'Segmentation':y_pred_test})

In [ ]:
submit_gnb['Segmentation']=submit_gnb['Segmentation'].apply(segment)

In [ ]:
submit_gnb.to_csv('janatahack.submit_gnb.csv',index=False)

10.KNN

In [ ]:
k_values=range(3,21)

In [ ]:
acc_score={}

In [ ]:
for k in k_values:
    knn=KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled,y_train)
    y_pred_train=knn.predict(X_train_scaled)
    y_pred_test=knn.predict(X_test_scaled)
    acc_score[k]=accuracy_score(y_train,y_pred_train)

In [ ]:
max(acc_score.values())

In [ ]:
max(acc_score,key=acc_score.get)

In [ ]:
#Rebuilding the model

In [ ]:
knn=KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train_scaled,y_train)
y_pred_train=knn.predict(X_train_scaled)
y_pred_test=knn.predict(X_test_scaled)

In [ ]:
print('Training Accuracy is: ',accuracy_score(y_train,y_pred_train))

In [ ]:
submit_knn=pd.DataFrame({'ID':df_test['ID'],'Segmentation':y_pred_test})

In [ ]:
submit_knn['Segmentation']=submit_knn['Segmentation'].apply(segment)

In [ ]:
submit_knn.to_csv('janatahack.submit_knn.csv',index=False)

11.SVC(kernel='linear')

In [ ]:
svc1=SVC(kernel='linear')

In [ ]:
svc1.fit(X_train_scaled,y_train)

In [ ]:
y_pred_train=svc1.predict(X_train_scaled)

In [ ]:
y_pred_test=svc1.predict(X_test_scaled)

In [ ]:
print('Training Accuracy is: ',accuracy_score(y_train,y_pred_train))

In [ ]:
submit_svc1=pd.DataFrame({'ID':df_test['ID'],'Segmentation':y_pred_test})

In [ ]:
submit_svc1['Segmentation']=submit_svc1['Segmentation'].apply(segment)

In [ ]:
submit_svc1.to_csv('janatahack.submit_svc1.csv',index=False)

12.SVC(kernel='radial')

In [ ]:
svc2=SVC(kernel='rbf')

In [ ]:
svc2.fit(X_train_scaled,y_train)

In [ ]:
y_pred_train=svc2.predict(X_train_scaled)

In [ ]:
y_pred_test=svc2.predict(X_test_scaled)

In [ ]:
print('Training Accuracy is: ',accuracy_score(y_train,y_pred_train))

In [ ]:
submit_svc2=pd.DataFrame({'ID':df_test['ID'],'Segmentation':y_pred_test})

In [ ]:
submit_svc2['Segmentation']=submit_svc2['Segmentation'].apply(segment)

In [ ]:
submit_svc2.to_csv('janatahack.submit_svc2.csv',index=False)

13.SVC(kernel='sigmoid')

In [ ]:
svc3=SVC(kernel='sigmoid')

In [ ]:
svc3.fit(X_train_scaled,y_train)

In [ ]:
y_pred_train=svc3.predict(X_train_scaled)

In [ ]:
y_pred_test=svc3.predict(X_test_scaled)

In [ ]:
print('Training Accuracy is: ',accuracy_score(y_train,y_pred_train))

In [ ]:
submit_svc3=pd.DataFrame({'ID':df_test['ID'],'Segmentation':y_pred_test})

In [ ]:
submit_svc3['Segmentation']=submit_svc3['Segmentation'].apply(segment)

In [ ]:
submit_svc3.to_csv('janatahack.submit_svc3.csv',index=False)

14.SVC(kernel='poly')

In [ ]:
svc4=SVC(kernel='poly')

In [ ]:
svc4.fit(X_train_scaled,y_train)

In [ ]:
y_pred_train=svc4.predict(X_train_scaled)

In [ ]:
y_pred_test=svc4.predict(X_test_scaled)

In [ ]:
print('Training Accuracy is: ',accuracy_score(y_train,y_pred_train))

In [ ]:
submit_svc4=pd.DataFrame({'ID':df_test['ID'],'Segmentation':y_pred_test})

In [ ]:
submit_svc4['Segmentation']=submit_svc4['Segmentation'].apply(segment)

In [ ]:
submit_svc4.to_csv('janatahack.submit_svc4.csv',index=False)

15.GridSearch CV with SVC (poly)

In [ ]:
tuned_parameters={'degree':[2,4,6],'gamma':['auto','scale'],'C':[0.25,0.5,1]}

In [ ]:
gs_model=GridSearchCV(estimator=SVC(kernel='poly'),param_grid=tuned_parameters,scoring='accuracy',cv=5)
gs_model.fit(X_train_scaled,y_train)

In [ ]:
print('Best Parameters are: ',gs_model.best_params_)

In [ ]:
svc5=SVC(kernel='poly',degree=4,gamma='auto',C=1)

In [ ]:
svc5.fit(X_train_scaled,y_train)

In [ ]:
y_pred_train=svc5.predict(X_train_scaled)

In [ ]:
y_pred_test=svc5.predict(X_test_scaled)

In [ ]:
print('Training Accuracy is: ',accuracy_score(y_train,y_pred_train))

In [ ]:
submit_svc5=pd.DataFrame({'ID':df_test['ID'],'Segmentation':y_pred_test})

In [ ]:
submit_svc5['Segmentation']=submit_svc5['Segmentation'].apply(segment)

In [ ]:
submit_svc5.to_csv('janatahack.submit_svc5.csv',index=False)

We can see that DecisionTreeClassifier and RandomForestClassifier performs better as compared to other models in terms of train accuracy.